# Fraudulent Transaction Detection

**Tabular Classification Project**

## 2 · Project Overview

Detect **fraudulent credit card transactions** from transaction-level features such as amount, merchant category, time of day, and cardholder demographics. The dataset has ~10,000 transactions with ~3% fraud rate — a classic imbalanced binary classification problem.

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given transaction details (amount, merchant category, time, location) and cardholder information (age, gender, card type), predict whether the transaction is fraudulent (is_fraud = 1).

## 5 · Why This Project Matters

- Global card fraud losses exceeded **$30 billion** in 2023.
- Real-time fraud detection saves billions annually for banks and consumers.
- Severe class imbalance (~3% fraud) makes this a challenging ML problem.
- This teaches precision-recall trade-offs critical for production fraud systems.
- False positives block legitimate purchases; false negatives lose money.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | ~10,000 |
| Features | 12 (amount, merchant_category, hour, day_of_week, age, gender, card_type, distance_from_home, distance_from_last_txn, ratio_to_median_amount, is_foreign, is_online) |
| Target | `is_fraud` (binary: 0=legitimate, 1=fraud) |
| Class balance | ~97% legitimate, ~3% fraud |
| Missing values | None |

## 7 · Dataset Source and License Notes

**Source**: Synthetic financial transaction dataset for ML practice.
**License**: Educational / public.
**Note**: Mimics credit card transaction patterns with realistic class imbalance.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "is_fraud"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

Save dir: E:\Github\Machine-Learning-Projects\Classification\Fraudulent Transaction Detection


## 11 · Dataset Download or Loading

In [4]:
np.random.seed(SEED)
n = 10000
fraud_rate = 0.03
n_fraud = int(n * fraud_rate)
n_legit = n - n_fraud

# Legitimate transactions
amt_legit = np.random.lognormal(3.5, 1.2, n_legit).clip(1, 5000).round(2)
amt_fraud = np.random.lognormal(5.0, 1.5, n_fraud).clip(50, 25000).round(2)
amount = np.concatenate([amt_legit, amt_fraud])

merchant_cats = ['grocery', 'gas_station', 'restaurant', 'online_retail', 'travel', 'entertainment', 'electronics', 'other']
merch_legit = np.random.choice(merchant_cats, n_legit, p=[0.25, 0.15, 0.15, 0.15, 0.05, 0.1, 0.05, 0.1])
merch_fraud = np.random.choice(merchant_cats, n_fraud, p=[0.05, 0.05, 0.05, 0.35, 0.15, 0.1, 0.2, 0.05])
merchant_category = np.concatenate([merch_legit, merch_fraud])

hour_legit = np.random.randint(0, 24, n_legit)
hour_fraud = np.random.randint(0, 24, n_fraud)
hour = np.concatenate([hour_legit, hour_fraud])
day_of_week = np.random.randint(0, 7, n)
age = np.random.normal(45, 15, n).clip(18, 85).astype(int)
gender = np.random.choice(['M', 'F'], n)
card_type = np.random.choice(['visa', 'mastercard', 'amex', 'discover'], n, p=[0.4, 0.35, 0.15, 0.1])

dist_home_legit = np.random.exponential(10, n_legit).clip(0, 200).round(1)
dist_home_fraud = np.random.exponential(80, n_fraud).clip(5, 5000).round(1)
distance_from_home = np.concatenate([dist_home_legit, dist_home_fraud])

dist_last_legit = np.random.exponential(5, n_legit).clip(0, 100).round(1)
dist_last_fraud = np.random.exponential(50, n_fraud).clip(1, 3000).round(1)
distance_from_last_txn = np.concatenate([dist_last_legit, dist_last_fraud])

ratio_legit = np.random.lognormal(0, 0.5, n_legit).clip(0.1, 10).round(2)
ratio_fraud = np.random.lognormal(1.5, 0.8, n_fraud).clip(1, 50).round(2)
ratio_to_median = np.concatenate([ratio_legit, ratio_fraud])

foreign_legit = np.random.binomial(1, 0.05, n_legit)
foreign_fraud = np.random.binomial(1, 0.35, n_fraud)
is_foreign = np.concatenate([foreign_legit, foreign_fraud])

online_legit = np.random.binomial(1, 0.25, n_legit)
online_fraud = np.random.binomial(1, 0.65, n_fraud)
is_online = np.concatenate([online_legit, online_fraud])

is_fraud = np.array([0]*n_legit + [1]*n_fraud)

df = pd.DataFrame({
    'amount': amount, 'merchant_category': merchant_category, 'hour': hour,
    'day_of_week': day_of_week, 'age': age, 'gender': gender, 'card_type': card_type,
    'distance_from_home': distance_from_home, 'distance_from_last_txn': distance_from_last_txn,
    'ratio_to_median_amount': ratio_to_median, 'is_foreign': is_foreign,
    'is_online': is_online, 'is_fraud': is_fraud,
})
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"Dataset shape: {df.shape}")
print(f"Fraud rate: {df['is_fraud'].mean():.2%}")
df.head()

Dataset shape: (10000, 13)
Fraud rate: 3.00%


,amount,merchant_category,hour,day_of_week,age,gender,card_type,distance_from_home,distance_from_last_txn,ratio_to_median_amount,is_foreign,is_online,is_fraud
0,784.82,restaurant,19,2,48,M,discover,23.7,1.5,3.08,0,0,0
1,9.54,online_retail,14,5,33,F,mastercard,5.4,9.9,1.60,0,0,0
2,21.78,grocery,3,5,63,M,mastercard,1.0,17.3,0.85,0,0,0
3,33.18,grocery,20,4,35,F,visa,24.0,9.6,2.36,0,0,0
4,120.21,grocery,13,3,46,F,visa,4.6,1.3,0.77,0,0,0


## 12 · Data Validation Checks

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

DATA VALIDATION

Shape: (10000, 13)

Missing values:
Series([], dtype: int64)
No missing values.

Duplicate rows: 0

Target distribution:
is_fraud
0    9700
1     300
Name: count, dtype: int64

Dtypes:
amount                    float64
merchant_category          object
hour                        int32
day_of_week                 int32
age                         int64
gender                     object
card_type                  object
distance_from_home        float64
distance_from_last_txn    float64
ratio_to_median_amount    float64
is_foreign                  int32
is_online                   int32
is_fraud                    int64
dtype: object

Target 'is_fraud' confirmed.


## 13 · Exploratory Data Analysis

In [6]:
num_cols = df.select_dtypes(include='number').columns.drop(TARGET).tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, col in enumerate(['amount', 'distance_from_home', 'distance_from_last_txn', 'ratio_to_median_amount', 'hour', 'age']):
    ax = axes[i // 3, i % 3]
    df[df[TARGET]==0][col].hist(bins=30, ax=ax, alpha=0.6, label='Legit', color='steelblue')
    df[df[TARGET]==1][col].hist(bins=30, ax=ax, alpha=0.6, label='Fraud', color='coral')
    ax.set_title(col); ax.legend()
plt.suptitle("Feature Distributions by Fraud Status", fontsize=14)
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, col in enumerate(['merchant_category', 'card_type']):
    ct = pd.crosstab(df[col], df[TARGET], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=axes[i], color=['steelblue', 'coral'])
    axes[i].set_title(f"Fraud Rate by {col}")
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()
print(f"Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}")

Numeric: 9, Categorical: 3


## 14 · Target Analysis

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title("Transaction Fraud Distribution")
axes[0].set_xticklabels(['Legitimate (0)', 'Fraud (1)'], rotation=0)
df[TARGET].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['steelblue', 'coral'])
axes[1].set_title("Class Proportions"); axes[1].set_ylabel('')
plt.tight_layout(); plt.show()
print(f"Class distribution:\n{df[TARGET].value_counts()}")
print(f"\nImbalance ratio: {df[TARGET].value_counts().iloc[0] / max(df[TARGET].value_counts().iloc[1], 1):.0f}:1")

Class distribution:
is_fraud
0    9700
1     300
Name: count, dtype: int64

Imbalance ratio: 32:1


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [8]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    df[cat_cols] = oe.fit_transform(df[cat_cols])

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().to_dict()}")

Train: (8000, 12), Test: (2000, 12)
Train target dist: {0: 7760, 1: 240}


## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [9]:
clean = [c.replace('-', '_').replace(' ', '_').replace('.', '_') for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (12): ['amount', 'merchant_category', 'hour', 'day_of_week', 'age', 'gender', 'card_type', 'distance_from_home', 'distance_from_last_txn', 'ratio_to_median_amount', 'is_foreign', 'is_online']


## 18 · Baseline Model

Logistic Regression as baseline.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
y_prob_bl = baseline.predict_proba(X_test)[:, 1] if len(np.unique(y_train)) == 2 else None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")

BASELINE: Logistic Regression
Accuracy : 0.9935
Precision: 0.9933
Recall   : 0.9935
F1       : 0.9933
ROC-AUC  : 0.9990


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision  Recall  Time Taken
Model                                                                                                        
Perceptron                       0.9940           0.972680  0.998325  0.994138   0.994404  0.9940    0.091130
GaussianNB                       0.9820           0.958419  0.995339  0.983637   0.987062  0.9820    0.087008
AdaBoostClassifier               0.9960           0.957560  0.999631  0.995967   0.995951  0.9960    0.465518
QuadraticDiscriminantAnalysis    0.9820           0.950344  0.995369  0.983542   0.986652  0.9820    0.471593
LGBMClassifier                   0.9970           0.950000  0.999777  0.996923   0.997009  0.9970    9.538873
XGBClassifier                    0.9955           0.949227  0.999416  0.995444   0.995426  0.9955    2.899801
CatBoostClassifier               0.9955           0.941151  0.999519  0.995405   0.995

## 20 · FLAML AutoML Run

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
_flaml_metric = "macro_f1" if y_train.nunique() > 2 else "f1"
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric=_flaml_metric,
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best: lgbm
Accuracy : 0.9960
F1       : 0.9959


## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [13]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test)
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost  Acc: 0.9965  F1: 0.9964


LightGBM  Acc: 0.9960  F1: 0.9959


XGBoost   Acc: 0.9975  F1: 0.9975


## 22 · Top 3 Model Selection

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
XGBoost                0.9975  0.9975     0.9975  0.9975
CatBoost               0.9965  0.9964     0.9965  0.9965
LightGBM               0.9960  0.9959     0.9960  0.9960
FLAML                  0.9960  0.9959     0.9960  0.9960
Logistic Regression    0.9935  0.9933     0.9933  0.9935

Top 3: ['XGBoost', 'CatBoost', 'LightGBM']


## 23 · Final Training and Evaluation of Top 3

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")
plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))


Detailed Classification Reports — Top 3:

  XGBoost
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1940
           1       0.98      0.93      0.96        60

    accuracy                           1.00      2000
   macro avg       0.99      0.97      0.98      2000
weighted avg       1.00      1.00      1.00      2000


  CatBoost
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1940
           1       1.00      0.88      0.94        60

    accuracy                           1.00      2000
   macro avg       1.00      0.94      0.97      2000
weighted avg       1.00      1.00      1.00      2000


  LightGBM
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1940
           1       0.98      0.88      0.93        60

    accuracy                           1.00      2000
   macro avg       0.99      0.94      0.96      20

## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]
errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

Best model: XGBoost
Error rate: 0.0025 (5 / 2000)

Errors by true class:
      errors  total  error_rate
true                           
0          1   1940    0.000515
1          4     60    0.066667


## 25 · Interpretation and Business Insight

- **Transaction amount** and **ratio to median** are the strongest fraud signals.
- **Distance from home** and **distance from last transaction** indicate card-not-present fraud.
- **Online** and **foreign** transactions have much higher fraud rates.
- **Late-night transactions** (midnight–6am) are disproportionately fraudulent.
- **Electronics** and **online retail** merchants have higher fraud concentration.

## 26 · Limitations

1. Synthetic data — real fraud patterns are more complex and adversarial.
2. Only 3% fraud rate — models may struggle with rare-event detection.
3. No temporal sequence — real fraud detection uses transaction sequences.
4. No cardholder behavioral history features.
5. Static feature set — real systems use hundreds of engineered features.

## 27 · How to Improve This Project

1. Apply class weights or SMOTE for better fraud recall.
2. Use PR-AUC as primary metric instead of accuracy.
3. Engineer velocity features (transactions per hour, per day).
4. Add sequence-based features from transaction history.
5. Implement threshold tuning for optimal precision-recall trade-off.

## 28 · Production Considerations

- Real-time scoring with <100ms latency requirements.
- Continuous model retraining as fraud patterns evolve.
- Human-in-the-loop review for borderline cases.
- Cost-sensitive optimization (fraud cost >> false positive cost).
- Regulatory compliance (explainability, fairness).

## 29 · Common Mistakes

1. Using accuracy on 97/3 imbalanced data (97% baseline!).
2. Not using class weights or resampling.
3. Ignoring the cost asymmetry (missed fraud >> false alarm).
4. Training on shuffled time-series data (temporal leakage).
5. Not monitoring for concept drift in production.

## 30 · Mini Challenge / Exercises

1. Calculate the accuracy of a 'predict all legitimate' dummy classifier.
2. Apply class_weight='balanced' and compare recall on fraud class.
3. Plot precision-recall curve and find the optimal threshold.
4. Compare fraud detection rate at 1% and 5% false positive rates.

## 31 · Final Summary / Key Takeaways

1. Fraud detection is a high-stakes imbalanced classification problem.
2. Accuracy is misleading — use F1, recall, PR-AUC.
3. Transaction amount, distance, and online/foreign flags are key signals.
4. Class weights and threshold tuning are essential for fraud recall.
5. Production systems need real-time scoring and continuous retraining.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }
with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))

Metrics saved.
{
  "CatBoost": {
    "accuracy": 0.9965,
    "f1": 0.9964,
    "precision": 0.9965,
    "recall": 0.9965
  },
  "LightGBM": {
    "accuracy": 0.996,
    "f1": 0.9959,
    "precision": 0.996,
    "recall": 0.996
  },
  "XGBoost": {
    "accuracy": 0.9975,
    "f1": 0.9975,
    "precision": 0.9975,
    "recall": 0.9975
  },
  "Logistic Regression": {
    "accuracy": 0.9935,
    "f1": 0.9933,
    "precision": 0.9933,
    "recall": 0.9935
  },
  "FLAML": {
    "accuracy": 0.996,
    "f1": 0.9959,
    "precision": 0.996,
    "recall": 0.996
  }
}
